# T39 - 成交数据
## 第四章：流动性分析

本章介绍债券市场流动性分析方法，包括：
1. 流动性指标计算
2. 流动性评分模型
3. 流动性分类
4. 流动性趋势分析

## 1. 导入必要的库

In [ ]:
# 数据处理
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import logging
from typing import Dict, List, Optional, Tuple

# 数据库连接
import sqlalchemy

# 导入配置
from config import config

# 配置日志
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger('流动性分析')

print('导入成功！')

## 2. 加载数据

In [ ]:
def load_trade_data(days: int = 60) -> pd.DataFrame:
    """加载成交数据"""
    engine = sqlalchemy.create_engine(
        config.database.get_mysql_yq_connection_string(),
        poolclass=sqlalchemy.pool.NullPool
    )
    
    end_date = datetime.now().strftime('%Y-%m-%d')
    start_date = (datetime.now() - timedelta(days=days)).strftime('%Y-%m-%d')
    
    query = f'''
        SELECT * FROM yq.cjqb 
        WHERE tradedDate >= '{start_date}' AND tradedDate <= '{end_date}'
    '''
    
    df = pd.read_sql(query, engine)
    
    # 数据预处理
    if not df.empty:
        df['tradedDate'] = pd.to_datetime(df['tradedDate'])
        df['dt'] = df['tradedDate'].dt.strftime('%Y-%m-%d')
        for col in ['tradedAmount', 'tradedPrice', 'tradedYield']:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
    
    print(f'加载 {len(df)} 条记录')
    return df

# 加载数据
df = load_trade_data(days=60)
print(f'日期范围: {df["dt"].min()} ~ {df["dt"].max()}')

## 3. 流动性指标计算

In [ ]:
class LiquidityAnalyzer:
    """流动性分析器"""
    
    def __init__(self, df: pd.DataFrame):
        self.df = df
    
    def calculate_trade_frequency(self) -> pd.DataFrame:
        """
        计算交易频率指标
        
        Returns:
            pd.DataFrame: 包含交易频率指标的数据
        """
        if self.df.empty:
            return pd.DataFrame()
        
        # 按债券统计
        freq_stats = self.df.groupby('bondCode').agg({
            'dt': 'nunique',  # 交易天数
            'tradedAmount': 'count'  # 交易次数
        }).reset_index()
        
        freq_stats.columns = ['bondCode', 'trade_days', 'trade_count']
        
        # 计算日均交易次数
        freq_stats['daily_trade_freq'] = freq_stats['trade_count'] / freq_stats['trade_days']
        
        # 计算交易频率评分 (0-1)
        max_freq = freq_stats['daily_trade_freq'].max()
        if max_freq > 0:
            freq_stats['freq_score'] = freq_stats['daily_trade_freq'] / max_freq
        else:
            freq_stats['freq_score'] = 0
        
        return freq_stats
    
    def calculate_volume_metrics(self) -> pd.DataFrame:
        """
        计算成交量指标
        
        Returns:
            pd.DataFrame: 包含成交量指标的数据
        """
        if self.df.empty:
            return pd.DataFrame()
        
        # 按债券统计
        vol_stats = self.df.groupby('bondCode').agg({
            'tradedAmount': ['sum', 'mean', 'std']
        }).reset_index()
        
        vol_stats.columns = ['bondCode', 'total_amount', 'avg_amount', 'std_amount']
        
        # 计算变异系数 (CV)
        vol_stats['cv'] = vol_stats['std_amount'] / vol_stats['avg_amount']
        vol_stats['cv'] = vol_stats['cv'].fillna(0)
        
        # 计算成交量评分 (0-1)
        max_amount = vol_stats['total_amount'].max()
        if max_amount > 0:
            vol_stats['volume_score'] = vol_stats['total_amount'] / max_amount
        else:
            vol_stats['volume_score'] = 0
        
        return vol_stats
    
    def calculate_price_volatility(self) -> pd.DataFrame:
        """
        计算价格波动率
        
        Returns:
            pd.DataFrame: 包含价格波动率的数据
        """
        if self.df.empty or 'tradedPrice' not in self.df.columns:
            return pd.DataFrame()
        
        # 按债券计算价格变化
        df_sorted = self.df.sort_values(['bondCode', 'tradedDate'])
        
        # 计算每日价格变化率
        df_sorted['price_return'] = df_sorted.groupby('bondCode')['tradedPrice'].pct_change()
        
        # 计算波动率
        vol_stats = df_sorted.groupby('bondCode').agg({
            'price_return': ['std', 'count']
        }).reset_index()
        
        vol_stats.columns = ['bondCode', 'price_volatility', 'price_count']
        vol_stats['price_volatility'] = vol_stats['price_volatility'].fillna(0)
        
        # 波动率越小，流动性越好（反向评分）
        max_vol = vol_stats['price_volatility'].max()
        if max_vol > 0:
            vol_stats['volatility_score'] = 1 - (vol_stats['price_volatility'] / max_vol)
        else:
            vol_stats['volatility_score'] = 1
        
        return vol_stats
    
    def calculate_yield_spread(self) -> pd.DataFrame:
        """
        计算收益率分布指标
        
        Returns:
            pd.DataFrame: 包含收益率分布指标的数据
        """
        if self.df.empty or 'tradedYield' not in self.df.columns:
            return pd.DataFrame()
        
        # 按债券统计收益率分布
        yield_stats = self.df.groupby('bondCode').agg({
            'tradedYield': ['mean', 'std', 'min', 'max']
        }).reset_index()
        
        yield_stats.columns = ['bondCode', 'avg_yield', 'yield_std', 'min_yield', 'max_yield']
        
        # 计算收益率范围
        yield_stats['yield_range'] = yield_stats['max_yield'] - yield_stats['min_yield']
        yield_stats['yield_std'] = yield_stats['yield_std'].fillna(0)
        
        # 收益率波动越小，流动性越好（反向评分）
        max_range = yield_stats['yield_range'].max()
        if max_range > 0:
            yield_stats['yield_score'] = 1 - (yield_stats['yield_range'] / max_range)
        else:
            yield_stats['yield_score'] = 1
        
        return yield_stats

# 创建分析器实例
analyzer = LiquidityAnalyzer(df)
print('流动性分析器初始化完成')

## 4. 计算各维度指标

In [ ]:
# 计算交易频率指标
freq_stats = analyzer.calculate_trade_frequency()
print('\n=== 交易频率指标 ===')
print(freq_stats.head(10).to_string())

In [ ]:
# 计算成交量指标
vol_stats = analyzer.calculate_volume_metrics()
print('\n=== 成交量指标 ===')
print(vol_stats.head(10).to_string())

In [ ]:
# 计算价格波动率
price_stats = analyzer.calculate_price_volatility()
print('\n=== 价格波动率 ===')
print(price_stats.head(10).to_string())

In [ ]:
# 计算收益率分布
yield_stats = analyzer.calculate_yield_spread()
print('\n=== 收益率分布 ===')
print(yield_stats.head(10).to_string())

## 5. 综合流动性评分

In [ ]:
def calculate_composite_liquidity_score(
    freq_stats: pd.DataFrame,
    vol_stats: pd.DataFrame,
    price_stats: pd.DataFrame,
    yield_stats: pd.DataFrame,
    weights: Dict[str, float] = None
) -> pd.DataFrame:
    """
    计算综合流动性评分
    
    Args:
        freq_stats: 交易频率指标
        vol_stats: 成交量指标
        price_stats: 价格波动率
        yield_stats: 收益率分布
        weights: 各指标权重
        
    Returns:
        pd.DataFrame: 综合流动性评分
    """
    if weights is None:
        weights = {
            'frequency': 0.3,   # 交易频率权重
            'volume': 0.3,      # 成交量权重
            'volatility': 0.2,  # 波动率权重
            'yield': 0.2        # 收益率权重
        }
    
    # 合并各维度指标
    composite = freq_stats[['bondCode', 'freq_score']].copy()
    
    if not vol_stats.empty:
        composite = composite.merge(
            vol_stats[['bondCode', 'volume_score']],
            on='bondCode',
            how='left'
        )
    
    if not price_stats.empty:
        composite = composite.merge(
            price_stats[['bondCode', 'volatility_score']],
            on='bondCode',
            how='left'
        )
    
    if not yield_stats.empty:
        composite = composite.merge(
            yield_stats[['bondCode', 'yield_score']],
            on='bondCode',
            how='left'
        )
    
    # 填充缺失值
    composite = composite.fillna(0)
    
    # 计算综合评分
    composite['liquidity_score'] = (
        composite['freq_score'] * weights['frequency'] +
        composite['volume_score'] * weights['volume'] +
        composite['volatility_score'] * weights['volatility'] +
        composite['yield_score'] * weights['yield']
    )
    
    # 归一化到0-100
    composite['liquidity_score'] = composite['liquidity_score'] * 100
    
    return composite

# 计算综合评分
liquidity_scores = calculate_composite_liquidity_score(
    freq_stats, vol_stats, price_stats, yield_stats
)

print('\n=== 综合流动性评分（前20名）===')
top_liquid = liquidity_scores.nlargest(20, 'liquidity_score')
print(top_liquid.to_string())

## 6. 流动性分类

In [ ]:
def classify_liquidity(df: pd.DataFrame) -> pd.DataFrame:
    """
    对债券进行流动性分类
    
    Args:
        df: 包含流动性评分的数据
        
    Returns:
        pd.DataFrame: 添加了流动性分类的数据
    """
    df = df.copy()
    
    # 使用配置中的阈值
    thresholds = config.trade_data.liquidity_thresholds
    
    # 将阈值转换为0-100范围
    high_threshold = thresholds['high'] * 100  # 70
    medium_threshold = thresholds['medium'] * 100  # 40
    
    # 分类
    conditions = [
        df['liquidity_score'] >= high_threshold,
        df['liquidity_score'] >= medium_threshold,
        True
    ]
    choices = ['高流动性', '中等流动性', '低流动性']
    
    df['liquidity_class'] = np.select(conditions, choices, default='低流动性')
    
    return df

# 进行流动性分类
liquidity_classified = classify_liquidity(liquidity_scores)

# 统计各分类数量
print('\n=== 流动性分类统计 ===')
class_stats = liquidity_classified['liquidity_class'].value_counts()
print(class_stats)
print(f'\n高流动性债券占比: {class_stats.get("高流动性", 0) / len(liquidity_classified) * 100:.1f}%')
print(f'中等流动性债券占比: {class_stats.get("中等流动性", 0) / len(liquidity_classified) * 100:.1f}%')
print(f'低流动性债券占比: {class_stats.get("低流动性", 0) / len(liquidity_classified) * 100:.1f}%')

## 7. 流动性趋势分析

In [ ]:
def analyze_liquidity_trend(df: pd.DataFrame) -> pd.DataFrame:
    """
    分析流动性趋势
    
    Args:
        df: 成交数据
        
    Returns:
        pd.DataFrame: 每日流动性指标
    """
    if df.empty:
        return pd.DataFrame()
    
    # 按日期聚合
    daily_stats = df.groupby('dt').agg({
        'bondCode': 'nunique',
        'tradedAmount': ['sum', 'count'],
        'tradedPrice': 'mean'
    }).reset_index()
    
    daily_stats.columns = ['dt', 'bond_count', 'total_amount', 'trade_count', 'avg_price']
    
    # 计算日均成交金额
    daily_stats['avg_trade_amount'] = daily_stats['total_amount'] / daily_stats['trade_count']
    
    # 计算移动平均
    daily_stats['amount_ma5'] = daily_stats['total_amount'].rolling(5, min_periods=1).mean()
    daily_stats['amount_ma20'] = daily_stats['total_amount'].rolling(20, min_periods=1).mean()
    
    # 计算趋势（与5日均线的偏离度）
    daily_stats['trend'] = (daily_stats['total_amount'] - daily_stats['amount_ma5']) / daily_stats['amount_ma5'] * 100
    
    return daily_stats

# 分析流动性趋势
trend_stats = analyze_liquidity_trend(df)

print('\n=== 流动性趋势分析 ===')
print(trend_stats.tail(20).to_string())

## 8. 流动性分析报告

In [ ]:
def generate_liquidity_report(
    liquidity_classified: pd.DataFrame,
    trend_stats: pd.DataFrame
):
    """生成流动性分析报告"""
    
    print('\n' + '='*70)
    print('流动性分析报告')
    print('='*70)
    
    # 1. 总体概览
    print('\n1. 总体概览')
    print(f'   分析债券数: {len(liquidity_classified):,}')
    print(f'   平均流动性评分: {liquidity_classified["liquidity_score"].mean():.2f}')
    print(f'   最高流动性评分: {liquidity_classified["liquidity_score"].max():.2f}')
    print(f'   最低流动性评分: {liquidity_classified["liquidity_score"].min():.2f}')
    
    # 2. 流动性分布
    print('\n2. 流动性分布')
    class_stats = liquidity_classified['liquidity_class'].value_counts()
    total = len(liquidity_classified)
    for cls in ['高流动性', '中等流动性', '低流动性']:
        count = class_stats.get(cls, 0)
        pct = count / total * 100
        print(f'   {cls}: {count:,} ({pct:.1f}%)')
    
    # 3. 高流动性债券
    print('\n3. 高流动性债券TOP10')
    high_liquidity = liquidity_classified[liquidity_classified['liquidity_class'] == '高流动性']
    top10 = high_liquidity.nlargest(10, 'liquidity_score')
    for i, row in top10.iterrows():
        print(f'   {row["bondCode"]}: 评分 {row["liquidity_score"]:.2f}')
    
    # 4. 流动性趋势
    if not trend_stats.empty:
        print('\n4. 流动性趋势')
        latest = trend_stats.iloc[-1]
        print(f'   最新日期: {latest["dt"]}')
        print(f'   成交债券数: {latest["bond_count"]}')
        print(f'   总成交金额: {latest["total_amount"]:,.0f}')
        print(f'   成交笔数: {latest["trade_count"]}')
        print(f'   趋势偏离度: {latest["trend"]:.2f}%')
        
        if latest['trend'] > 10:
            print('   趋势判断: 流动性上升')
        elif latest['trend'] < -10:
            print('   趋势判断: 流动性下降')
        else:
            print('   趋势判断: 流动性平稳')
    
    print('\n' + '='*70)

# 生成报告
generate_liquidity_report(liquidity_classified, trend_stats)